# 1. Importar librerías

In [1]:
import os
import re
import json
import base64
import getpass
import requests
import chromadb
import google.generativeai as genai
import time
from google.api_core.exceptions import ResourceExhausted

C:\Users\zaida\AppData\Local\Temp\ipykernel_5508\2925707296.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# 2. Configuración

In [2]:
GITHUB_USER = "zaaidaaraque"
EMBEDDING_MODEL = "models/gemini-embedding-001"
CHAT_MODEL = "gemini-3.6-flash"
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "portfolio_projects"

# API key gratuita de Google Gemini
gemini_api_key = getpass.getpass("Introduce tu Gemini API key: ")
genai.configure(api_key=gemini_api_key)

# Token de GitHub
github_token = getpass.getpass("Introduce tu GitHub token (recomendado, pulsa Enter para omitir): ")
GITHUB_HEADERS = {"Authorization": f"token {github_token}"} if github_token else {}

Introduce tu Gemini API key:  ········
Introduce tu GitHub token (recomendado, pulsa Enter para omitir):  ········


# 3. Listas repositorios públicos

In [3]:
def get_public_repos(username):
    repos = []
    page = 1
    while True:
        url = f"https://api.github.com/users/{username}/repos"
        params = {"per_page": 100, "page": page, "type": "owner"}
        resp = requests.get(url, headers=GITHUB_HEADERS, params=params)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        repos.extend(batch)
        page += 1
    return [r for r in repos if not r["fork"]]

repos = get_public_repos(GITHUB_USER)
print(f"Repos públicos encontrados: {len(repos)}")
for r in repos:
    print("-", r["name"])

Repos públicos encontrados: 9
- explanatory_models_breast_cancer
- ml_pump_it_up_driven_data
- nlp_disaster_tweets_kaggle
- power_bi_dashboard_netflix_analysis
- predictive_modeling_hotel_booking_cancellations
- scoring_model
- sql_smart_desk
- statistics_blood_glucose_analysis
- tableau_dashboard_easy_loans


# 4. Descargar README de cada repositorio
También se guarda el SHA del contenido para más adelante detectar cambios.

In [4]:
def get_readme(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    resp = requests.get(url, headers=GITHUB_HEADERS)
    if resp.status_code != 200:
        return None, None
    data = resp.json()
    content = base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
    sha = data["sha"]
    return content, sha

projects = []
for r in repos:
    content, sha = get_readme(r["full_name"])
    if content and len(content.strip()) > 0:
        projects.append({
            "name": r["name"],
            "full_name": r["full_name"],
            "description": r.get("description") or "",
            "url": r["html_url"],
            "readme": content,
            "sha": sha,
        })

print(f"Proyectos con README válido: {len(projects)}")

Proyectos con README válido: 9


# 5. Extraer contenido de los notebooks (.ipynb)

Se extrae el texto de las **celdas markdown** (contienen explicaciones y conclusiones) y las **librerías importadas** en las celdas de código (para saber qué técnicas/herramientas se usaron).

In [5]:
def get_repo_notebooks(repo_full_name):
    for branch in ("main", "master"):
        url = f"https://api.github.com/repos/{repo_full_name}/git/trees/{branch}?recursive=1"
        resp = requests.get(url, headers=GITHUB_HEADERS)
        if resp.status_code == 200:
            tree = resp.json().get("tree", [])
            return [item["path"] for item in tree if item["path"].endswith(".ipynb")]
    return []

def get_file_content(repo_full_name, path):
    url = f"https://api.github.com/repos/{repo_full_name}/contents/{path}"
    resp = requests.get(url, headers=GITHUB_HEADERS)
    if resp.status_code != 200:
        return None, None
    data = resp.json()
    content = base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
    return content, data["sha"]

def extract_notebook_summary(notebook_json_text):
    try:
        nb_data = json.loads(notebook_json_text)
    except json.JSONDecodeError:
        return "", []

    markdown_parts = []
    imports = set()
    import_pattern = re.compile(r"^\s*(?:import|from)\s+([a-zA-Z0-9_\.]+)", re.MULTILINE)

    for cell in nb_data.get("cells", []):
        source = "".join(cell.get("source", []))
        if cell.get("cell_type") == "markdown":
            markdown_parts.append(source)
        elif cell.get("cell_type") == "code":
            imports.update(import_pattern.findall(source))

    markdown_text = "\n\n".join(markdown_parts)
    return markdown_text, sorted(imports)

for p in projects:
    notebook_paths = get_repo_notebooks(p["full_name"])
    notebook_texts = []
    all_imports = set()

    for path in notebook_paths:
        content, _ = get_file_content(p["full_name"], path)
        if not content:
            continue
        md_text, imports = extract_notebook_summary(content)
        all_imports.update(imports)
        if md_text.strip():
            notebook_texts.append(f"### Notebook: {path}\n{md_text}")

    p["notebook_content"] = "\n\n".join(notebook_texts)
    p["libraries"] = sorted(all_imports)
    print(f"{p['name']}: {len(notebook_paths)} notebook(s) encontrado(s)")


explanatory_models_breast_cancer: 1 notebook(s) encontrado(s)
ml_pump_it_up_driven_data: 1 notebook(s) encontrado(s)
nlp_disaster_tweets_kaggle: 1 notebook(s) encontrado(s)
power_bi_dashboard_netflix_analysis: 0 notebook(s) encontrado(s)
predictive_modeling_hotel_booking_cancellations: 1 notebook(s) encontrado(s)
scoring_model: 1 notebook(s) encontrado(s)
sql_smart_desk: 0 notebook(s) encontrado(s)
statistics_blood_glucose_analysis: 1 notebook(s) encontrado(s)
tableau_dashboard_easy_loans: 0 notebook(s) encontrado(s)


# 6. Chunking del contenido
Se divide por encabezados Markdown en vez de por tamaño fijo de tokens.

In [6]:
def split_by_headers(text):
    pattern = r"(?=^#{1,3}\s)"
    sections = re.split(pattern, text, flags=re.MULTILINE)
    sections = [s.strip() for s in sections if s.strip()]
    return sections if sections else [text.strip()]

def build_chunks(projects):
    chunks = []
    for p in projects:
        sections = split_by_headers(p["readme"])
        for i, section in enumerate(sections):
            chunk_text = f"Proyecto: {p['name']}\n{p['description']}\n\n{section}"
            chunks.append({
                "id": f"{p['name']}__readme__{i}",
                "text": chunk_text,
                "metadata": {
                    "project": p["name"],
                    "url": p["url"],
                    "sha": p["sha"],
                    "source": "readme",
                }
            })

        libraries = p.get("libraries", [])
        if libraries:
            lib_text = (
                f"Proyecto: {p['name']}\n{p['description']}\n\n"
                f"Librerías y herramientas usadas en el código: {', '.join(libraries)}"
            )
            chunks.append({
                "id": f"{p['name']}__libraries",
                "text": lib_text,
                "metadata": {
                    "project": p["name"],
                    "url": p["url"],
                    "sha": p["sha"],
                    "source": "notebook_libraries",
                }
            })

        notebook_content = p.get("notebook_content", "")
        if notebook_content.strip():
            nb_sections = split_by_headers(notebook_content)
            for i, section in enumerate(nb_sections):
                chunk_text = f"Proyecto: {p['name']}\n{p['description']}\n\n{section}"
                chunks.append({
                    "id": f"{p['name']}__notebook__{i}",
                    "text": chunk_text,
                    "metadata": {
                        "project": p["name"],
                        "url": p["url"],
                        "sha": p["sha"],
                        "source": "notebook",
                    }
                })
    return chunks

chunks = build_chunks(projects)
print(f"Total de chunks generados: {len(chunks)}")

Total de chunks generados: 134


# 7. Embeddings con Gemini
Se usa el modelo de embeddings de Gemini y se guardan los vectores en una colección persistente de ChromaDB, junto con la metadata (proyecto, URL, sha).

El nivel gratuito de Gemini tiene un límite de peticiones de embeddings por minuto, así que se deja una pequeña pausa entre llamadas.

In [7]:
SECONDS_BETWEEN_CALLS = 0.7

def get_embedding(text, task_type="retrieval_document", max_retries=5):
    for attempt in range(max_retries):
        try:
            result = genai.embed_content(model=EMBEDDING_MODEL, content=text, task_type=task_type)
            return result["embedding"]
        except ResourceExhausted as e:
            wait = getattr(e, "retry_delay", None)
            wait_seconds = wait.seconds if wait else 5
            print(f"Límite de la API alcanzado, esperando {wait_seconds}s antes de reintentar...")
            time.sleep(wait_seconds + 1)
    raise RuntimeError("Se superó el número máximo de reintentos llamando a la API de embeddings.")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

def index_chunks(chunks):
    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk["text"], task_type="retrieval_document")
        collection.upsert(
            ids=[chunk["id"]],
            embeddings=[embedding],
            documents=[chunk["text"]],
            metadatas=[chunk["metadata"]],
        )
        time.sleep(SECONDS_BETWEEN_CALLS)
        if (i + 1) % 10 == 0:
            print(f"Indexados {i + 1}/{len(chunks)} chunks...")

index_chunks(chunks)
print(f"Chunks indexados en ChromaDB: {collection.count()}")

Indexados 10/134 chunks...
Indexados 20/134 chunks...
Indexados 30/134 chunks...
Indexados 40/134 chunks...
Indexados 50/134 chunks...
Indexados 60/134 chunks...
Indexados 70/134 chunks...
Indexados 80/134 chunks...
Indexados 90/134 chunks...
Indexados 100/134 chunks...
Indexados 110/134 chunks...
Indexados 120/134 chunks...
Indexados 130/134 chunks...
Chunks indexados en ChromaDB: 134


# 8. Retrieval y generación

In [8]:
def retrieve(question, k=4):
    query_embedding = get_embedding(question, task_type="retrieval_query")
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    return [
        {"text": doc, "metadata": meta}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

gemini_model = genai.GenerativeModel(CHAT_MODEL)

def ask(question, k=4, verbose_sources=True):
    retrieved = retrieve(question, k=k)
    context = "\n\n---\n\n".join(r["text"] for r in retrieved)

    prompt = (
        "Eres un asistente que responde preguntas sobre los proyectos "
        "de Zaida, basándote únicamente en el contexto proporcionado. "
        "Si la respuesta no está en el contexto, dilo claramente en vez de inventar. "
        "Cuando menciones un proyecto, indica su nombre.\n\n"
        f"Contexto:\n{context}\n\nPregunta: {question}"
    )

    response = gemini_model.generate_content(prompt)
    answer = response.text

    if verbose_sources:
        sources = sorted(set(r["metadata"]["project"] for r in retrieved))
        print("Fuentes usadas:", ", ".join(sources))
        for r in retrieved:
            print(f"  - {r['metadata']['project']}: {r['metadata']['url']}")
        print()

    return answer

# 9. Testear Chatbot

In [9]:
print(ask("¿Qué proyectos de NLP has hecho y qué técnicas usaste?"))

Fuentes usadas: nlp_disaster_tweets_kaggle
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle

Basándome en el contexto proporcionado, el proyecto de NLP realizado es:

**Proyecto:** `nlp_disaster_tweets_kaggle`

---

### **Técnicas y modelos utilizados en el proyecto `nlp_disaster_tweets_kaggle`:**

1. **Técnicas de preprocesamiento de texto:**
   - Conversión a minúsculas (*Lowercase conversion*)
   - Eliminación de URLs
   - Eliminación de etiquetas HTML
   - Eliminación de caracteres de escape
   - Eliminación de puntuación
   - Eliminación de números
   - Eliminación de palabras de parada (*Stopword removal*)
   - Lematización (*Lemmatization*)
   - Eliminación de espacios 

In [10]:
print(ask("¿Qué proyectos de visualización de datos tiene?"))

Fuentes usadas: power_bi_dashboard_netflix_analysis, tableau_dashboard_easy_loans
  - tableau_dashboard_easy_loans: https://github.com/zaaidaaraque/tableau_dashboard_easy_loans
  - power_bi_dashboard_netflix_analysis: https://github.com/zaaidaaraque/power_bi_dashboard_netflix_analysis
  - power_bi_dashboard_netflix_analysis: https://github.com/zaaidaaraque/power_bi_dashboard_netflix_analysis
  - power_bi_dashboard_netflix_analysis: https://github.com/zaaidaaraque/power_bi_dashboard_netflix_analysis

Basándome en el contexto proporcionado, Zaida cuenta con los siguientes proyectos de visualización de datos:

1. **`tableau_dashboard_easy_loans`**: Un panel interactivo en Tableau que analiza visualmente las operaciones de préstamos, la actividad de los comerciantes y el comportamiento de los reembolsos de Easy Loans para el año 2023 en diferentes países.
2. **`power_bi_dashboard_netflix_analysis`**: Un panel interactivo en Power BI que analiza casi 10,000 películas (con datos en formato T

In [11]:
print(ask("¿Qué técnicas de preprocesamiento de datos usas normalmente?"))

Fuentes usadas: ml_pump_it_up_driven_data, nlp_disaster_tweets_kaggle, scoring_model
  - ml_pump_it_up_driven_data: https://github.com/zaaidaaraque/ml_pump_it_up_driven_data
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle
  - nlp_disaster_tweets_kaggle: https://github.com/zaaidaaraque/nlp_disaster_tweets_kaggle
  - scoring_model: https://github.com/zaaidaaraque/scoring_model

Basándome en los proyectos proporcionados, las técnicas de preprocesamiento de datos varían según el tipo de proyecto y problema:

*   **En el proyecto `ml_pump_it_up_driven_data`:**
    *   Imputación de valores faltantes.
    *   Eliminación de características irrelevantes.
    *   Manejo de características categóricas.
    *   Creación de diferentes subconjuntos de características.
    *   Validación cruzada.
    *   Evaluación y prueba de versiones del conjunto de datos con variables originales e imputadas.

*   **En el proyecto `nlp_disaster_tweets_kaggle`:**
    *  

## 10. Detección de cambios
Para no tener que actualizar el índice a mano, esta función compara el `sha` del README actual de cada repo con el que ya está guardado en ChromaDB. Solo reprocesa (y vuelve a generar embeddings) los proyectos que hayan cambiado o sean nuevos.

In [12]:
def get_indexed_shas():
    all_data = collection.get(include=["metadatas"])
    shas = {}
    for meta in all_data["metadatas"]:
        shas[meta["project"]] = meta["sha"]
    return shas

def sync_index():
    current_repos = get_public_repos(GITHUB_USERNAME)
    indexed_shas = get_indexed_shas()

    updated_projects = []
    for r in current_repos:
        content, sha = get_readme(r["full_name"])
        if not content:
            continue
        if indexed_shas.get(r["name"]) != sha:
            notebook_paths = get_repo_notebooks(r["full_name"])
            notebook_texts, all_imports = [], set()
            for path in notebook_paths:
                nb_content, _ = get_file_content(r["full_name"], path)
                if not nb_content:
                    continue
                md_text, imports = extract_notebook_summary(nb_content)
                all_imports.update(imports)
                if md_text.strip():
                    notebook_texts.append(f"### Notebook: {path}\n{md_text}")

            updated_projects.append({
                "name": r["name"],
                "full_name": r["full_name"],
                "description": r.get("description") or "",
                "url": r["html_url"],
                "readme": content,
                "sha": sha,
                "notebook_content": "\n\n".join(notebook_texts),
                "libraries": sorted(all_imports),
            })

    if not updated_projects:
        print("No hay cambios: el índice ya está actualizado.")
        return

    new_chunks = build_chunks(updated_projects)
    index_chunks(new_chunks)
    print(f"Actualizados {len(updated_projects)} proyecto(s): "
          f"{', '.join(p['name'] for p in updated_projects)}")
